# 3.10.5 Unemployment Rate (LUR) 실업률 경제 지표 분석

- 데이터: `WEOApr2024.xlsx`
- code: **LUR**
- 2022년 국가별 실업률 **오름차순 80개국**
- 관심 국가(Korea, USA, Japan, Germany) 표시
- 같은 그래프에 **2023 − 2022** 차이값 표시

VS Code에서 이 파일을 연 뒤, 위쪽 **Run All** 또는 셀마다 `Shift+Enter`로 실행하세요.


In [ ]:
# 필요 라이브러리
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

plt.rcParams["axes.unicode_minus"] = False


## 1) WEO 파일에서 LUR(실업률) 불러오기

In [ ]:
# 노트북과 같은 폴더의 WEOApr2024.xlsx 사용
weo_path = Path("WEOApr2024.xlsx")
if not weo_path.exists():
    weo_path = Path("lur_analysis") / "WEOApr2024.xlsx"

raw = pd.read_excel(weo_path)
lur = raw[raw["WEO Subject Code"] == "LUR"].copy()

# 2022, 2023 숫자로 변환
lur[2022] = pd.to_numeric(lur[2022], errors="coerce")
lur[2023] = pd.to_numeric(lur[2023], errors="coerce")

df = lur[["ISO", "Country", 2022, 2023]].dropna().copy()
df = df.rename(columns={"ISO": "country_code", "Country": "country", 2022: "LUR_2022", 2023: "LUR_2023"})
df = df[df["country_code"].str.fullmatch(r"[A-Z]{3}")].reset_index(drop=True)

print("LUR 데이터 국가 수:", len(df))
df.head()


## 2) 2022년 실업률 오름차순 정렬 (80개국) + 2023−2022 차이

In [ ]:
# 관심 국가 3~4개
FOCUS = ["KOR", "USA", "JPN", "DEU"]

analysis = df.copy()
analysis["diff_2023_2022"] = analysis["LUR_2023"] - analysis["LUR_2022"]  # 2023 - 2022
analysis = analysis.sort_values("LUR_2022", ascending=True).head(80).reset_index(drop=True)
analysis["focus"] = analysis["country_code"].isin(FOCUS)

print("분석 국가 수:", len(analysis))
analysis.head(10)


## 3) 관심 국가 확인

In [ ]:
analysis[analysis["focus"]][["country_code", "country", "LUR_2022", "LUR_2023", "diff_2023_2022"]]


## 4) 하나의 그래프에 표시
- 막대: 2022년 실업률
- 선: 2023 − 2022 차이
- 빨강: 관심 국가

In [ ]:
fig, ax1 = plt.subplots(figsize=(18, 8))

x = range(len(analysis))
colors = ["#D62728" if f else "#4C78A8" for f in analysis["focus"]]

# 2022 실업률 (오름차순 80개국)
ax1.bar(x, analysis["LUR_2022"], color=colors, alpha=0.85, label="2022 LUR (%)")
ax1.set_ylabel("2022 Unemployment Rate (LUR, %)")
ax1.set_xlabel("Countries (ascending by 2022 LUR)")
ax1.set_xticks(list(x))
ax1.set_xticklabels(analysis["country"], rotation=90, fontsize=7)
ax1.grid(axis="y", linestyle="--", alpha=0.3)

# 2023 - 2022 차이 (같은 그래프, 오른쪽 축)
ax2 = ax1.twinx()
ax2.plot(
    list(x),
    analysis["diff_2023_2022"],
    color="#2CA02C",
    marker="o",
    linewidth=1.5,
    markersize=3.5,
    label="2023 - 2022 (pp)",
)
ax2.axhline(0, color="gray", linewidth=1, linestyle=":")
ax2.set_ylabel("Difference 2023 - 2022 (percentage points)")

# 관심 국가 라벨
for i, row in analysis[analysis["focus"]].iterrows():
    ax1.annotate(
        row["country"],
        xy=(i, row["LUR_2022"]),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color="#D62728",
    )
    ax2.scatter([i], [row["diff_2023_2022"]], color="#D62728", s=55, zorder=5)

ax1.set_title(
    "IMF WEO LUR | 2022 Unemployment Rate (Ascending, 80 countries)\n"
    "with 2023 - 2022 Difference | source: WEOApr2024.xlsx"
)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2 + [Patch(color="#D62728")], l1 + l2 + ["Focus countries"], loc="upper left")

fig.tight_layout()
plt.show()
